In [1]:

from langchain_core.documents import Document
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os

os.makedirs("../data/text_files", exist_ok=True)

sample_texts = {
    "../data/text_files/python_intro.txt": """Python Programming Introduction
Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular programming languages in the world.
Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
Python is widely used in web development, data science, artificial intelligence, and automation.""",

    "../data/text_files/machine_learning.txt": """Machine Learning Basics
Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed.
Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties
Applications include image recognition, speech processing, and recommendation systems."""
}

for filepath, content in sample_texts.items():
    with open(filepath, 'w', encoding="utf-8") as f:
        f.write(content)

dir_loader = DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={'encoding': 'utf-8'}
)
documents = dir_loader.load()

/var/folders/51/9gx7md516190yyxnj102pv2w0000gn/T/ipykernel_97740/3513522386.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader, DirectoryLoader
/Users/shreyanshshailesh/Desktop/Retrieval Augmented Generation Basic/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

def split_documents(documents):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    return split_docs

chunks = split_documents(documents)

Split 2 documents into 2 chunks


In [3]:

import numpy as np
from typing import List
from sentence_transformers import SentenceTransformer

class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = SentenceTransformer(self.model_name)
        print(f"Model loaded. Dimension: {self.model.get_sentence_embedding_dimension()}")

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        return self.model.encode(texts, show_progress_bar=True)

embedding_manager = EmbeddingManager()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7519.29it/s]


Model loaded. Dimension: 384


/var/folders/51/9gx7md516190yyxnj102pv2w0000gn/T/ipykernel_97740/3118545889.py:10: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded. Dimension: {self.model.get_sentence_embedding_dimension()}")


In [4]:

import chromadb
import uuid
from typing import List, Any, Dict

class VectorStore:
    def __init__(self, collection_name: str = "pdf_documents"):
        self.chroma_client = chromadb.PersistentClient(path="../data/vector_store")
        self.collection = self.chroma_client.get_or_create_collection(name=collection_name)
        print(f"Collection: {collection_name}, Docs: {self.collection.count()}")

    def add_documents(self, documents, embeddings):
        ids, metadatas, docs_text, embeddings_list = [], [], [], []
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            ids.append(f"doc_{uuid.uuid4().hex[:8]}_{i}")
            metadata = dict(doc.metadata) if hasattr(doc, 'metadata') else {}
            metadata['doc_index'] = i
            metadatas.append(metadata)
            docs_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())
        self.collection.add(ids=ids, embeddings=embeddings_list, metadatas=metadatas, documents=docs_text)
        print(f"Added {len(documents)} docs. Total: {self.collection.count()}")

vectorstore = VectorStore()

# Add chunks to vector store
texts = [chunk.page_content for chunk in chunks]
embeddings = embedding_manager.generate_embeddings(texts)
vectorstore.add_documents(chunks, embeddings)

Collection: pdf_documents, Docs: 0


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.91s/it]

Added 2 docs. Total: 2


In [5]:

class RAGRetriever:
    def __init__(self, vector_store, embedding_manager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5) -> List[Dict]:
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        results = self.vector_store.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_k
        )
        retrieved_docs = []
        if results['documents'] and results['documents'][0]:
            for i, (doc_id, document, metadata, distance) in enumerate(zip(
                results['ids'][0], results['documents'][0],
                results['metadatas'][0], results['distances'][0]
            )):
                retrieved_docs.append({
                    'id': doc_id,
                    'content': document,
                    'metadata': metadata,
                    'similarity_score': 1 - distance,
                    'rank': i + 1
                })
        return retrieved_docs

rag_retriever = RAGRetriever(vectorstore, embedding_manager)

In [7]:

from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")  # no space before GROQ_API_KEY

llm = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="llama-3.1-8b-instant",
    temperature=0.1,
    max_tokens=1024
)

def rag_simple(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc.get('content', '') for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    prompt = f"""Use the following context to answer the question concisely.
Context:
{context}
Question: {query}
Answer: """
    response = llm.invoke([prompt])
    return response.content

In [8]:

answer = rag_simple("how does sentiment analysis work", rag_retriever, llm)
print(answer)

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.77it/s]


Sentiment analysis is a type of supervised learning task that uses machine learning algorithms to determine the emotional tone or sentiment behind a piece of text, such as positive, negative, or neutral. 

Here's a concise overview of how sentiment analysis works:

1. **Data Collection**: A large dataset of labeled text samples is collected, where each sample is associated with its sentiment (positive, negative, or neutral).
2. **Preprocessing**: The text data is cleaned and preprocessed to remove noise, punctuation, and special characters.
3. **Feature Extraction**: The preprocessed text data is converted into numerical features, such as word frequencies or word embeddings (e.g., Word2Vec or GloVe).
4. **Model Training**: A machine learning model, such as a neural network or a decision tree, is trained on the labeled dataset to learn the patterns and relationships between the text features and sentiment labels.
5. **Model Evaluation**: The trained model is evaluated on a separate test